# NYC 311 Bronze Ingestion Pipeline 

### Purpose
This pipeline ingests NYC 311 Service Request data from the Socrata Open Data API, validates schema and data quality, applies deduplication logic, and writes results into Delta tables. It is designed as a **batch-only ingestion** process with auditability and provenance tracking.

---

### Phase 1: Setup
- Initializes Spark session and runtime identifiers (`ingest_run_id`, `today`, `now`).
- Defines Lakehouse paths for Bronze, Silver, and Gold layers.
- Creates directories using Fabric’s file system API to ensure consistent storage structure.

---

### Phase 2: Smart Fetch with Retry
- Implements `fetch_with_retry` function:
  - Retries API calls up to 3 times with exponential backoff.
  - Handles transient network/API errors gracefully.
- Determines which months are already ingested by querying the Bronze table.
- Fetches only **missing months + current month**, avoiding redundant downloads.
- Saves results as **JSONL snapshots** (one record per line) to handle free-text safely.

---

### Phase 3: Validation & Deduplication
- **Validation (`validate_311_bronze`)**:
  - Reads raw JSONL into Spark.
  - Flattens nested struct columns.
  - Casts fields into correct types (timestamps, doubles, integers, strings).
  - Flags missing critical fields (`unique_key`, `created_date`, `agency`, `status`, `borough`).
  - Splits into **good records** vs. **quarantine records**.
  - Adds provenance metadata (`source`, `ingest_run_id`, `ingested_at`, `ingest_date`).

- **Intra-Batch Deduplication**:
  - Drops duplicates within the new batch on `(unique_key, status)`.

- **7-Day Deduplication Against Bronze Table**:
  - Uses a window function to get the latest status per complaint (`unique_key`).
  - Buckets records into:
    - **New** (not in bronze yet),
    - **Updated** (status changed),
    - **Skipped** (status unchanged).
  - Only **new + updated** records are appended.

---

### Phase 3b: Write to Delta Tables
- Good records are appended to `bronze_311_nyc` with partitioning by `ingest_date`.
- Invalid records are appended to `bronze_quarantine_311_nyc`.
- Append-only writes maintain auditability and schema evolution support.

---

### Phase 4: QA Queries
The pipeline includes SQL queries to validate ingestion quality and provide audit trails:
1. **Record integrity**: total rows, distinct complaints, ingestion runs.  
2. **Agency distribution**: top 10 agencies by complaint count.  
3. **Borough distribution**: record counts by borough.  
4. **Status distribution**: counts per complaint status.  
5. **Null checks**: verifies critical fields are populated.  
6. **Date range**: earliest and latest complaint/closure dates.  
7. **Audit trail**: complaints with multiple status changes.  
8. **Status progression**: ordered timeline of complaint statuses.  
9. **Dedup effectiveness**: measures how many duplicates were skipped vs. status changes retained.


### Phase 1: set-up

In [3]:
import datetime, uuid, requests, json, time

from functools import reduce
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, StringType
from pyspark.sql import functions as F
from datetime import date
from notebookutils import mssparkutils
from pyspark.sql import types as T
from datetime import date


from pyspark.sql import SparkSession
# Initialize Spark
spark = SparkSession.builder.getOrCreate()

#runtime identifiers
now = datetime.datetime.utcnow()
ingest_run_id = str(uuid.uuid4())


StatementMeta(, d2662a64-43fe-48e1-928a-a7945f6a16f0, 4, Finished, Available, Finished, False)

In [4]:
LAKEHOUSE = "test_lakehouse_1"   
BASE_PATH = "Files/project_data"  

PATHS = {
    "bronze_weather": f"{BASE_PATH}/bronze/weather",
    "bronze_311":     f"{BASE_PATH}/bronze/311",
    "bronze_traffic": f"{BASE_PATH}/bronze/traffic",
    "silver":         f"{BASE_PATH}/silver",
    "gold":           f"{BASE_PATH}/gold",
}

# Create directories using Fabric's file system API with relative paths
for name, path in PATHS.items():
    try:
        mssparkutils.fs.mkdirs(path)
        #print(f" {name}: {path}")
    except Exception as e:
        print(f"{name} creation issue: {e}")

print(f"\nAll folders ready under {BASE_PATH}")

StatementMeta(, d2662a64-43fe-48e1-928a-a7945f6a16f0, 5, Finished, Available, Finished, False)


All folders ready under Files/project_data


In [15]:
before_count = spark.table("bronze_311_nyc").count()
print(f"Before: {before_count:,} rows")

StatementMeta(, d2662a64-43fe-48e1-928a-a7945f6a16f0, 16, Finished, Available, Finished, False)

Before: 1,226,417 rows


### Phase 2: Smart Fetch with Retry

In [16]:
today = datetime.date.today()
today_str = today.isoformat()
year = 2026

def fetch_with_retry(url, params, max_retries=3, backoff=5):
    """
    Perform an HTTP GET request with retry logic.

    Args:
        url (str): The API endpoint to call.
        params (dict): Query parameters for the request.
        max_retries (int): Maximum number of retry attempts (default = 3).
        backoff (int): Delay in seconds between retries (default = 5).

    Returns:
        dict: JSON response from the API if successful.

    Raises:
        Exception: If all retry attempts fail.
    """
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, params=params, timeout=60)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                time.sleep(backoff)
    raise Exception("All retries failed")

# Determine which months are already fully ingested
already_ingested_months = set()
try:
    df_existing = spark.table("bronze_311_nyc")
    existing_count = df_existing.count()
    ingested_months = df_existing.select(
        F.year("created_date").alias("yr"),
        F.month("created_date").alias("mn")
    ).distinct().collect()
    # Only skip past months — current month always re-fetched (still accumulating)
    for row in ingested_months:
        if (row['yr'], row['mn']) < (today.year, today.month):
            already_ingested_months.add((row['yr'], row['mn']))
    print(f"Existing bronze_311_nyc: {existing_count:,} rows")
    print(f"Skipping already-ingested past months: {sorted(already_ingested_months)}")
except Exception:
    print("No existing table — fetching full history.")

# Fetch only missing months + current month
BASE_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
all_records = []
current_month = today.month if today.year == year else 12

for month in range(1, current_month + 1):
    if (year, month) in already_ingested_months:
        print(f"Skipping {year}-{month:02d} — already ingested")
        continue

    start_date = datetime.date(year, month, 1)
    end_date = datetime.date(year + 1, 1, 1) if month == 12 else datetime.date(year, month + 1, 1)
    offset = 0
    print(f"Fetching {year}-{month:02d} ({start_date} to {end_date})...")

    while True:
        params = {
            "$limit": 50000,
            "$offset": offset,
            "$where": f"created_date >= '{start_date}T00:00:00' AND created_date < '{end_date}T00:00:00'",
            "$order": "created_date ASC"
        }
        try:
            batch = fetch_with_retry(BASE_URL, params)
        except Exception as e:
            print(f"  Failed at offset {offset}: {e}")
            break
        if not batch:
            break
        all_records.extend(batch)
        offset += 50000
        print(f"  {year}-{month:02d}: {len(all_records):,} records so far...")
        if len(batch) < 50000:
            print(f"  {year}-{month:02d} complete")
            break

print(f"\nTotal new records fetched: {len(all_records):,}")

# Save as JSONL (one record per line) 
# JSONL avoids all CSV quoting/delimiter issues with free-text fields like
# addresses and complaint descriptions that contain embedded commas and quotes.
if all_records:
    jsonl_path = f"{PATHS['bronze_311']}/nyc_311_{today_str}.jsonl"
    jsonl_content = "\n".join(json.dumps(record) for record in all_records)
    mssparkutils.fs.put(jsonl_path, jsonl_content, overwrite=True)
    file_size_mb = len(jsonl_content.encode('utf-8')) / 1024 / 1024
    print(f"Saved: {jsonl_path}")
    print(f"File size: {file_size_mb:.1f} MB | Records: {len(all_records):,}")
    new_data_fetched = True
else:
    jsonl_path = None
    new_data_fetched = False
    print("No new records to fetch — already up to date.")


StatementMeta(, d2662a64-43fe-48e1-928a-a7945f6a16f0, 17, Finished, Available, Finished, False)

Existing bronze_311_nyc: 1,226,417 rows
Skipping already-ingested past months: [(2026, 1), (2026, 2), (2026, 3)]
Skipping 2026-01 — already ingested
Skipping 2026-02 — already ingested
Skipping 2026-03 — already ingested
Fetching 2026-04 (2026-04-01 to 2026-05-01)...
  2026-04: 50,000 records so far...
  2026-04: 100,000 records so far...
  2026-04: 150,000 records so far...
  2026-04: 200,000 records so far...
  2026-04: 212,278 records so far...
  2026-04 complete

Total new records fetched: 212,278
Saved: Files/project_data/bronze/311/nyc_311_2026-04-23.jsonl
File size: 299.2 MB | Records: 212,278


### Phase 3: Validation

In [17]:
def validate_311_bronze(jsonl_path, ingest_run_id):
    """
    Validate and preprocess NYC 311 JSONL data for Bronze layer ingestion.

    Args:
        jsonl_path (str): Path to the raw JSONL file containing 311 records.
        ingest_run_id (str): Unique identifier for the ingestion run.

    Returns:
        tuple: (good, quarantine)
            - good (DataFrame): Valid records ready for Bronze table.
            - quarantine (DataFrame): Invalid records flagged for review.
    """

    print(f"\n[BRONZE] Reading raw JSONL: {jsonl_path}")

    df = spark.read.json(jsonl_path)

    raw_count = df.count()
    print(f"  Raw rows: {raw_count:,}")

    # Flatten any nested struct columns to top-level
    from pyspark.sql.types import StructType
    nested_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StructType)]
    for col_name in nested_cols:
        sub_fields = [f.name for f in df.schema[col_name].dataType.fields]
        for sub in sub_fields:
            df = df.withColumn(f"{col_name}_{sub}", F.col(f"{col_name}.{sub}"))
        df = df.drop(col_name)

    # Type casting
    df = (df
        .withColumn("created_date", F.col("created_date").cast("timestamp"))
        .withColumn("closed_date", F.col("closed_date").cast("timestamp"))
        .withColumn("due_date", F.col("due_date").cast("timestamp"))
        .withColumn("resolution_action_updated_date", F.col("resolution_action_updated_date").cast("date"))
        .withColumn("latitude", F.col("latitude").cast(DoubleType()))
        .withColumn("longitude", F.col("longitude").cast(DoubleType()))
        .withColumn("council_district", F.col("council_district").cast(IntegerType()))
        .withColumn("incident_zip", F.col("incident_zip").cast(StringType()))
        .withColumn("bbl", F.col("bbl").cast(StringType()))
    )

    # Null checks on critical fields 
    df = df.withColumn("flag_missing_key",     F.col("unique_key").isNull())
    df = df.withColumn("flag_missing_created", F.col("created_date").isNull())
    df = df.withColumn("flag_missing_agency",  F.col("agency").isNull())
    df = df.withColumn("flag_missing_status",  F.col("status").isNull())
    df = df.withColumn("flag_missing_borough", F.col("borough").isNull())

    flag_names = [c for c in df.columns if c.startswith("flag_")]
    combined_flags = reduce(lambda acc, c: acc | F.col(c), flag_names, F.lit(False))
    df = df.withColumn("is_valid", ~combined_flags)

    # Provenance metadata
    df = (df
        .withColumn("source",        F.lit("nyc-open-data-311"))
        .withColumn("ingest_run_id", F.lit(ingest_run_id))
        .withColumn("ingested_at",   F.current_timestamp())
        .withColumn("ingest_date",   F.lit(date.today().isoformat()))
    )

    good  = df.filter(F.col("is_valid") == True).drop("is_valid", *flag_names)
    quarantine = df.filter(F.col("is_valid") == False)

    print(f"  Good rows:       {good.count():,}")
    print(f"  Quarantine rows: {quarantine.count():,}")

    return good, quarantine


StatementMeta(, d2662a64-43fe-48e1-928a-a7945f6a16f0, 18, Finished, Available, Finished, False)

### Phase 3: Deduplication

In [20]:
# Phase 3: Validate + Write to Delta (with 7-day dedup) 
if not new_data_fetched:
    print("No new data to ingest — skipping.")
else:
    ingest_run_id = "run_" + str(uuid.uuid4())

    good, quarantine = validate_311_bronze(jsonl_path, ingest_run_id)

    good_count_raw   = good.count()
    quarantine_count = quarantine.count()
    print(f"Validated — Good: {good_count_raw:,} | Quarantine: {quarantine_count:,}")

    # Lightweight dedup guard within the new batch on (unique_key, status)
    good       = good.dropDuplicates(["unique_key", "status"])
    good_count = good.count()
    if good_count < good_count_raw:
        print(f"Removed {good_count_raw - good_count:,} intra-batch duplicates on (unique_key, status)")
    print(f"New good rows to append: {good_count:,}")

# Phase 3a: 7-Day Dedup Logic (Using bronze_311_nyc)
# Compares new batch against bronze_311_nyc table to filter unchanged records
# Dedup key: (unique_key, status) — removes exact duplicates

if new_data_fetched and good_count > 0:
    print("\n" + "="*80)
    print("APPLYING 7-DAY DEDUP LOGIC (bronze_311_nyc)")
    print("="*80)
    
    try:
        df_existing_bronze = spark.table("bronze_311_nyc")
        existing_count = df_existing_bronze.count()
        
        if existing_count > 0:
            print(f"\nExisting bronze_311_nyc: {existing_count:,} rows")
            
            # Get latest status from bronze_311_nyc (per unique_key) 
            # So the window lets you: Group by unique_key (so you know which rows belong to the same complaint), 
            # Sort by created_date DESC (so rank 1 = newest), Keep only rank 1 (so you get the latest status for each complaint)
            window_spec = Window.partitionBy("unique_key").orderBy(F.desc("created_date"))
            
            df_bronze_latest = (
                df_existing_bronze
                .withColumn("rn", F.row_number().over(window_spec))
                .filter(F.col("rn") == 1)
                .select("unique_key", F.col("status").alias("_bronze_status"))
            )
            
            # Join new batch to latest bronze status 
            df_joined = good.join(df_bronze_latest, on="unique_key", how="left")
            
            # Define 3 bucket masks
            mask_new = F.col("_bronze_status").isNull()
            
            mask_updated = (~F.col("_bronze_status").isNull()) & (F.col("status") != F.col("_bronze_status"))
            
            mask_skipped = (~F.col("_bronze_status").isNull()) & (F.col("status") == F.col("_bronze_status"))
            
            # Split into 3 buckets 
            df_bucket_new = df_joined.filter(mask_new).drop("_bronze_status")
            df_bucket_updated = df_joined.filter(mask_updated).drop("_bronze_status")
            df_bucket_skipped = df_joined.filter(mask_skipped).drop("_bronze_status")
            
            n_new = df_bucket_new.count()
            n_updated = df_bucket_updated.count()
            n_skipped = df_bucket_skipped.count()
            
            print(f"\n 7-Day Dedup Results:")
            print(f"    New rows (not in bronze):       {n_new:,}")
            print(f"    Updated rows (status changed):  {n_updated:,}")
            print(f"    Skipped (status unchanged):    {n_skipped:,}")
            print(f"    Total validated:                {good_count:,}")
            
            count_check = n_new + n_updated + n_skipped
            status_check = " matches" if count_check == good_count else " MISMATCH"
            print(f"\nValidation: {n_new} + {n_updated} + {n_skipped} = {count_check} ({status_check})")
            
            # Only write new + updated (skip unchanged) 
            good = df_bucket_new.union(df_bucket_updated)
            good_count_deduped = good.count()
            
            if good_count != good_count_deduped:
                rows_skipped = good_count - good_count_deduped
                print(f"\n Dedup Impact:")
                print(f"   Before dedup: {good_count:,} rows")
                print(f"   After dedup:  {good_count_deduped:,} rows")
                print(f"   Skipped:      {rows_skipped:,} unchanged records")
                good_count = good_count_deduped  # Update for subsequent checks
            else:
                print(f"\n No unchanged records detected (all new or status changed)")
        else:
            print(f"Bronze_311_nyc table empty — writing all {good_count:,} validated records")
            
    except Exception as e:
        print(f"\n  7-day dedup skipped due to error: {e}")
        print(f"Proceeding with write of {good_count:,} records")


# Phase 3b: Write to Table (APPEND ONLY)
    if good_count > 0:
        print("\n" + "="*80)
        print("WRITING TO bronze_311_nyc (APPEND MODE)")
        print("="*80)
        
        print(f"Before write:")
        print(f"  bronze_311_nyc: {existing_count:,} rows")
        
        print(f"\nAppending {good_count:,} records (new + updated)...")
        good.write \
            .format("delta") \
            .mode("append") \
            .partitionBy("ingest_date")\
            .option("mergeSchema", "true") \
            .saveAsTable("bronze_311_nyc")
        
        new_count = spark.table("bronze_311_nyc").count()
        rows_added = new_count - existing_count
        
        print(f"\nAfter write:")
        print(f"  bronze_311_nyc: {new_count:,} rows")
        print(f"  Rows added: {rows_added:,}")
        
        print(f"\nWrite Validation:")
        print(f"  Expected to add: {good_count:,} rows")
        print(f"  Actually added:  {rows_added:,} rows")
        
        if rows_added == good_count:
            print(f"   PASS — Write count matches!")
        else:
            print(f"   FAIL — Write count MISMATCH!")

    if quarantine_count > 0:
        print(f"\nWriting {quarantine_count:,} quarantine rows to bronze_quarantine_311_nyc...")
        quarantine.write \
            .format("delta") \
            .mode("append") \
            .partitionBy("ingest_date")\
            .option("mergeSchema", "true") \
            .saveAsTable("bronze_quarantine_311_nyc")
        print("Quarantine table updated")

    print(f"\n311 ingestion complete! Run ID: {ingest_run_id}")


StatementMeta(, d2662a64-43fe-48e1-928a-a7945f6a16f0, 21, Finished, Available, Finished, False)


[BRONZE] Reading raw JSONL: Files/project_data/bronze/311/nyc_311_2026-04-23.jsonl
  Raw rows: 212,278
  Good rows:       212,278
  Quarantine rows: 0
Validated — Good: 212,278 | Quarantine: 0
New good rows to append: 212,278

APPLYING 7-DAY DEDUP LOGIC (bronze_311_nyc)

Existing bronze_311_nyc: 1,242,380 rows

 7-Day Dedup Results:
    New rows (not in bronze):       0
    Updated rows (status changed):  4,828
    Skipped (status unchanged):    207,450
    Total validated:                212,278

Validation: 0 + 4828 + 207450 = 212278 ( matches)

 Dedup Impact:
   Before dedup: 212,278 rows
   After dedup:  4,828 rows
   Skipped:      207,450 unchanged records

WRITING TO bronze_311_nyc (APPEND MODE)
Before write:
  bronze_311_nyc: 1,242,380 rows

Appending 4,828 records (new + updated)...

After write:
  bronze_311_nyc: 1,247,208 rows
  Rows added: 4,828

Write Validation:
  Expected to add: 4,828 rows
  Actually added:  4,828 rows
   PASS — Write count matches!

311 ingestion compl

##### Debug checks to ensure the number of rows added

In [21]:
after_count = spark.table("bronze_311_nyc").count()
print(f"After: {after_count:,} rows")
print(f"Added: {after_count - before_count:,} rows")

StatementMeta(, d2662a64-43fe-48e1-928a-a7945f6a16f0, 22, Finished, Available, Finished, False)

After: 1,247,208 rows
Added: 20,791 rows


### Phase 4: QA Queries

In [22]:
# QA QUERIES FOR BRONZE LAYER - 311

# 1. Total count and record integrity
spark.sql("""
    SELECT 
        COUNT(*) as total_records,
        COUNT(DISTINCT unique_key) as unique_complaints,
        COUNT(DISTINCT ingest_run_id) as ingestion_runs
    FROM bronze_311_nyc
""").show()

# 2. Records by agency (top 10)
print("\nTop 10 agencies by complaint count:")
spark.sql("""
    SELECT 
        agency,
        COUNT(*) as complaint_count
    FROM bronze_311_nyc
    GROUP BY agency
    ORDER BY complaint_count DESC
    LIMIT 10
""").show()

# 3. Records by borough
print("\nRecords by borough:")
spark.sql("""
    SELECT 
        borough,
        COUNT(*) as record_count
    FROM bronze_311_nyc
    GROUP BY borough
    ORDER BY record_count DESC
""").show()

# 4. Status distribution
print("\nComplaint status distribution:")
spark.sql("""
    SELECT 
        status,
        COUNT(*) as count
    FROM bronze_311_nyc
    GROUP BY status
    ORDER BY count DESC
""").show()

# 5. Null checks in critical fields
print("\nNull checks (should all be 0):")
spark.sql("""
    SELECT 
        SUM(CASE WHEN unique_key IS NULL THEN 1 ELSE 0 END) as null_unique_key,
        SUM(CASE WHEN created_date IS NULL THEN 1 ELSE 0 END) as null_created_date,
        SUM(CASE WHEN agency IS NULL THEN 1 ELSE 0 END) as null_agency,
        SUM(CASE WHEN status IS NULL THEN 1 ELSE 0 END) as null_status,
        SUM(CASE WHEN borough IS NULL THEN 1 ELSE 0 END) as null_borough
    FROM bronze_311_nyc
""").show()

# 6. Date range
print("\nDate range:")
spark.sql("""
    SELECT 
        MIN(created_date) as earliest_complaint,
        MAX(created_date) as latest_complaint,
        MIN(closed_date) as earliest_closure,
        MAX(closed_date) as latest_closure
    FROM bronze_311_nyc
""").show()

# 7. AUDIT TRAIL: Show complaints with multiple status changes
print("\nComplaints with status changes (audit trail):")
spark.sql("""
    SELECT 
        unique_key,
        COUNT(*) as status_change_count,
        COLLECT_LIST(STRUCT(status, ingested_at)) as status_history
    FROM bronze_311_nyc
    GROUP BY unique_key
    HAVING COUNT(*) > 1
    ORDER BY status_change_count DESC
    LIMIT 20
""").show(truncate=False)

# 8. Same unique_key with different statuses (ordered by time)
print("\nSame unique_key showing status progression:")
spark.sql("""
    SELECT 
        unique_key,
        status,
        created_date,
        ingested_at,
        ingest_run_id
    FROM bronze_311_nyc
    WHERE unique_key IN (
        SELECT unique_key
        FROM bronze_311_nyc
        GROUP BY unique_key
        HAVING COUNT(*) > 1
    )
    ORDER BY unique_key, ingested_at
    LIMIT 50
""").show(truncate=False)

# 9. Dedup effectiveness: How many duplicates were skipped?
print("\nDedup effectiveness:")
spark.sql("""
    WITH complaint_counts AS (
        SELECT 
            unique_key,
            COUNT(*) as versions
        FROM bronze_311_nyc
        GROUP BY unique_key
    )
    SELECT 
        COUNT(*) as unique_complaints,
        SUM(versions) as total_rows,
        SUM(CASE WHEN versions = 1 THEN 1 ELSE 0 END) as unchanged_records,
        SUM(CASE WHEN versions > 1 THEN versions - 1 ELSE 0 END) as status_changes,
        ROUND(100.0 * SUM(CASE WHEN versions > 1 THEN versions - 1 ELSE 0 END) / SUM(versions), 2) as pct_changed
    FROM complaint_counts
""").show()

StatementMeta(, d2662a64-43fe-48e1-928a-a7945f6a16f0, 23, Finished, Available, Finished, False)

+-------------+-----------------+--------------+
|total_records|unique_complaints|ingestion_runs|
+-------------+-----------------+--------------+
|      1247208|          1237552|             3|
+-------------+-----------------+--------------+


Top 10 agencies by complaint count:
+------+---------------+
|agency|complaint_count|
+------+---------------+
|  NYPD|         468725|
|   HPD|         356362|
|  DSNY|         136134|
|   DOT|         108073|
|   DEP|          61383|
|   DOB|          34564|
|   DPR|          23753|
| DOHMH|          23048|
|   TLC|          11348|
|   DHS|          11157|
+------+---------------+


Records by borough:
+-------------+------------+
|      borough|record_count|
+-------------+------------+
|     BROOKLYN|      381167|
|       QUEENS|      304403|
|        BRONX|      261367|
|    MANHATTAN|      240049|
|STATEN ISLAND|       59081|
|  Unspecified|        1141|
+-------------+------------+


Complaint status distribution:
+-----------+-------+
